### Hybrid Retrival Argumented Generation Evelution using RAGAS 

In [2]:
import warnings 
warnings.filterwarnings('ignore')

# Document load 
from langchain_community.document_loaders import PyPDFLoader 
loader  = PyPDFLoader('Static GK 2025.pdf')
pages = loader.load()

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import hashlib

# Split Data 

spliter = RecursiveCharacterTextSplitter(chunk_size=1400 , chunk_overlap=180)
text_spliter = spliter.split_documents(pages)
chunks = [i.page_content for i in text_spliter]
metadata = [i.metadata for i in text_spliter]
ids = [hashlib.md5(chunk.encode('utf-8')).hexdigest() for chunk in chunks]
print(f'print first 5 ids : {ids[:2]}')

print first 5 ids : ['df52eef7bfa55759b4642211e13e3020', '622d6c3b19974d6f39f9950848df1607']


In [4]:
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 

embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# client and collection create 
client = chromadb.PersistentClient(path="./Hybrid_RAG")
collection = client.get_or_create_collection(name="Hybrid_RAG",embedding_function=embedding_function)

if chunks:
    collection.add(
        ids=ids,
        documents=chunks , metadatas=metadata
    )
collection.count()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

225

In [5]:
from langchain_ollama import ChatOllama 
llm = ChatOllama(model="qwen2.5:1.5b")

In [6]:
# Hybrid Corpus 
from rank_bm25 import BM25Okapi 
def tokenization(token):
    token = token.lower()
    token = token.split()
    return token 

tokens = [tokenization(i) for i in chunks]
bm_corpus = BM25Okapi(tokens)

print(f'sucussfully : {bm_corpus}')

sucussfully : <rank_bm25.BM25Okapi object at 0x10fb15550>


In [7]:
def Hybrid_Retrive(query:str):
    query_re = llm.invoke(f"write the query based on symentic search : {query}").content.strip()

    # Thats Vector DB retrival 
    result = collection.query(query_texts=[query_re] , n_results=5)
    dis  = result['distances'][0] 
    docs = result['documents'][0]
    threshold = 0.9
    print(f'the distance is : {dis}')
    dense_docs = []
    for i , d in zip(dis,docs):
        if threshold > i :
            dense_docs.append(d)
    # Thats Hybrid RAG Retrival using indexing 
    query_tokens = tokenization(query_re)
    score = bm_corpus.get_scores(query=query_tokens)
    def get_top_tokens (score , k=10):
        index = list(enumerate(score))
        idx_sorted = sorted(index,key=lambda x:x[1],reverse=True)
        return [doc for doc , _ in idx_sorted[:10]]
    index_tokens = [chunks[i] for i in get_top_tokens(score=score,k=10)]
    
    rrf_token = {}
    
    for rank , doc in enumerate(dense_docs):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_tokens):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
        
    marge = sorted(rrf_token.items() , key = lambda x:x[1] , reverse=True)
    get_docs = [i for i , _ in marge[:5]]
    
    return get_docs
    
def generation_answer(question:str , context_list:list):
    if not context_list :
        return "NOT Related Content"
    content_str = "\n\n".join(context_list) 
    
    prompt = f""" 
    Give answer based on the local document , if cant find out any related content 
    then direct type NOT related content 
    content : {content_str}
    question :{question}
    """
    response = llm.invoke(prompt)
    
    return response.content

In [9]:
# RAG evelute 
from datasets import Dataset 
from ragas import evaluate 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
user_input = []
retrival_context = []
response = []
reference = []

test_cases = [
    {
        "question": "who is first First Chief of Army Staff",
        "ground_truth": "General Maharaj Rajendra Singh Ji was the first Chief of Army Staff."
    },
    {
        "question": "Where is Malhargad Fort located?",
        "ground_truth": "Malhargad Fort is located in Sonori, near Saswad, Pune district, Maharashtra."
    },
    {
        "question": "Who built the Red Fort in Delhi?",
        "ground_truth": "Red Fort was built by Mughal Emperor Shah Jahan in 1648 AD."
    },
    {
        "question": "What is Purandar Fort famous for?",
        "ground_truth": "Purandar Fort is famous as the birthplace of Chhatrapati Sambhaji Maharaj."
    },
    {
        "question": "Which dynasty built Chitradurga Fort originally?",
        "ground_truth": "Chitradurga Fort was originally built by the Chalukyas between the 11th and 13th centuries."
    }
]
for item in test_cases:
    q =  item['question']
    truth = item['ground_truth']
    
    context = Hybrid_Retrive(query=q)
    LLM_answer = generation_answer(question=q , context_list=context)
    
    user_input.append(q)
    retrival_context.append(context)
    response.append(LLM_answer)
    reference.append(truth)
    
    data = {
        "user_input":user_input , 
        "retrieved_contexts":retrival_context , 
        "response":response , 
        "reference":reference
    }
    data = Dataset.from_dict(data)
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print(LLM_answer)

result = evaluate(
dataset= data, 
metrics=[Faithfulness(),AnswerRelevancy(),ContextPrecision(),ContextRecall()],
embeddings=embeddings , 
llm=llm,
raise_exceptions=False
)

df = result.to_pandas()
print(df)

the distance is : [0.6201715469360352, 0.6818681359291077, 0.7067350149154663, 0.7150964736938477, 0.733440101146698]
the distance is : [0.5094835162162781, 0.5523144006729126, 0.5666491389274597, 0.6315736174583435, 0.6360076665878296]
the distance is : [0.6249947547912598, 0.6636049747467041, 0.6854329109191895, 0.6868695020675659, 0.6919662356376648]
the distance is : [0.5959161520004272, 0.6048527956008911, 0.6055808067321777, 0.6107357144355774, 0.6373057961463928]
the distance is : [0.3439257740974426, 0.49520421028137207, 0.5029313564300537, 0.5074663758277893, 0.5167422294616699]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

The original dynasty that built Chitradurga Fort originally was the Chalukyas.


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

                                         user_input  ... context_recall
0            who is first First Chief of Army Staff  ...            1.0
1                  Where is Malhargad Fort located?  ...            1.0
2                  Who built the Red Fort in Delhi?  ...            1.0
3                 What is Purandar Fort famous for?  ...            1.0
4  Which dynasty built Chitradurga Fort originally?  ...            1.0

[5 rows x 8 columns]


In [ ]:
!uv pip show ragas

Using Python 3.12.13 environment at: /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv
Name: ragas
Version: 0.4.3
Location: /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv/lib/python3.12/site-packages
Requires: appdirs, datasets, diskcache, instructor, langchain, langchain-community, langchain-core, langchain-openai, nest-asyncio, networkx, numpy, openai, pillow, pydantic, rich, scikit-network, tiktoken, tqdm, typer
Required-by:
